In [1]:
import pandas as pd

df = pd.read_csv("data/augmented_df.csv")
texts = df["symptoms"].tolist()

len(texts)

10020

In [2]:
# LOAD DistilBert

from transformers import DistilBertTokenizerFast, DistilBertModel
import torch
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
model = DistilBertModel.from_pretrained("distilbert-base-uncased").to(device)
model.eval()

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSdpaAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [3]:
# Encode Text in Batches

batch_size = 16
bert_embeddings = []

for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]

    enc = tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        cls_vectors = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        bert_embeddings.append(cls_vectors)

bert_embeddings = np.vstack(bert_embeddings)
bert_embeddings.shape


(10020, 768)

In [4]:
# save bert features

import os
os.makedirs("data/features", exist_ok=True)

np.save("data/features/X_bert.npy", bert_embeddings)

print("Saved → data/features/X_bert.npy")

Saved → data/features/X_bert.npy


In [5]:
print("Samples:", bert_embeddings.shape[0])
print("Embedding size:", bert_embeddings.shape[1])

Samples: 10020
Embedding size: 768


In [6]:
import numpy as np

np.save("data/features/bert_embeddings.npy", bert_embeddings)
print("Saved → data/features/bert_embeddings.npy")

Saved → data/features/bert_embeddings.npy
